In [ ]:
import requests
from xml.dom import minidom
import csv
import sys
import io

def get_node_text(node_list):
    """Lấy nội dung văn bản từ node đầu tiên trong danh sách."""
    if node_list and node_list[0].firstChild:
        # Sử dụng nodeValue để đảm bảo lấy được nội dung (kể cả từ CDATA)
        return node_list[0].firstChild.nodeValue.strip()
    return ''

# --- Bước 1: Tải RSS feed từ URL và lưu thành file XML ---
url = 'http://www.hindustantimes.com/rss/topnews/rssfeed.xml'
xml_filename = 'rss_feed.xml'
csv_filename = 'tin_tuc.csv'

# Giả lập trình duyệt để tránh bị chặn
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/100.0.4896.127 Safari/537.36'
}

print("Bắt đầu tải RSS feed...")
try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() 

    with open(xml_filename, 'w', encoding='utf-8') as f:
        f.write(response.text)
    print(f" Đã lưu RSS feed thành {xml_filename}")

except requests.exceptions.RequestException as e:
    print(f" Lỗi tải RSS feed: {e}")
    sys.exit()

tin_tuc_list = []

try:
   
    
    with open(xml_filename, 'r', encoding='utf-8') as f:
        xml_content = f.read()
    
    doc = minidom.parseString(xml_content)
    items = doc.getElementsByTagName('item')

    for item in items:
        tin = {}
        
        tin['title'] = get_node_text(item.getElementsByTagName('title'))
        tin['link'] = get_node_text(item.getElementsByTagName('link'))
        tin['pubDate'] = get_node_text(item.getElementsByTagName('pubDate'))
        
        tin_tuc_list.append(tin)

    print(f" Đã phân tích {len(tin_tuc_list)} mục tin tức.")

except FileNotFoundError:
    print(f" Lỗi: Không tìm thấy tệp {xml_filename}.")
    sys.exit()
except Exception as e:
   
    print(f" Lỗi phân tích XML: {e}")
    sys.exit()

if tin_tuc_list:
    try:
        with open(csv_filename, 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['title', 'link', 'pubDate']
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            writer.writerows(tin_tuc_list)

        print(f" Đã lưu tin tức vào {csv_filename}")
    except IOError as e:
        print(f" Lỗi ghi file CSV: {e}")
else:
    print(" Không có tin tức nào để lưu vào CSV.")